# Week 08 - Topic Modeling - Latent Dirichlet Allocation

------

Welcome back! Today, we'll work on a very useful text analysis technique called Topic Modeling. Particularly, we will use its most popular and powerful algorithm: Latent Dirichlet Allocation, or LDA

Before you implement your own LDA on Pyro, it is important that you understand all concepts well. In this notebook, we'll start by playing a little bit with the Dirichlet distribution. After this, we'll ask you to do some ancestral sampling on the LDA generative story.

Only after that, you'll do your own LDA in Pyro. 


We start by the usual imports

In [47]:
import torch
from torch import nn
from torch.distributions import constraints
import functools

import pyro
import pyro.distributions as dist
from pyro.infer import SVI, JitTraceEnum_ELBO, TraceEnum_ELBO
from pyro.contrib.autoguide import AutoDiagonalNormal, AutoMultivariateNormal, AutoGuideList, AutoDelta
from pyro.optim import ClippedAdam

In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

----------------

## Part 2: LDA Ancestral Sampling

We define our own dictionary of 30 words. There are 3 different topics that we embedded in the dictionary, can you see what they are?

In [49]:
K = 3 # Offensive Magic, Defensive Magic, Utility Magic
# Spells are now being sorted into their dominant category.
# A spell may be mostly defensive, but there is an overlap between defensive, and utility.
dictionary =['Killing Curse', # avada kedavra
             'Blasting Curse', # Confringo
             'Cruciatus Curse', # Crucio
             'Leg-Locker Curse', # Locomotor Mortis
             'Imperius Curse', # Imperio
             'Full Body-Bind Curse', # Petrificus Totalus
             'Exploding Charm', # Bombarda
             'Expulso Curse', # Expulso
             'Memory Charm', # Obliviate
             'Oppugno Jinx', # Oppugno
             'Sectumsempra', # Sectumsempra
             'Stunning Spell', # Stupefy
             
             'Disarming Charm', # Expelliarmus
             'Slowing Charm', # Arresto Momentum
             'Brackium Emendo',  # Healing Spell
             'Episkey', # Healing Spell
             'Patronus Charm', # Expecto Patronum
             'General Counter-Spell', # Finite
             'Freezing Charm', # Glacius
             'Impediment Jinx', # Impedimenta
             'Reverse Spell', # Priori Incantatem
             'Shield Charm', # Protego
             'Protego Maxima', # Protego Maxima
             'Protego totalum', # Protego Totalum
             'Revulsion Jinx', # Relashio
             'Mending Charm', # Reparo
             'Repello Inimicum', # Repello Inimicum
             'Salvio hexia', # Salvio Hexia
             
             'Summoning Charm', # Accio
             'Water-Making Spell', # Aguamenti
             'Launch an object up into the air', # Alarte Ascendare
             'Unlocking Charm', # Alohomora
             'Spider repelling spell', # Arania Exumai
             'Cistem Aperio', # Cistem Aperio
             'Locking Spell', # Colloportus
             'Severing Charm', # Diffindo
             'Dissendium', # Dissendium
             'Engorgement Charm', # Engorgio
             'Human Presence Revealing Spell', # Homenum Revelio 
             'Incarcerous Spell', # Incarcerous
             'Fire-Making Spell', # Incendio
             'Levicorpus', # Levicorpus
             'Locomotion Charm', # Locomotor
             'Wand-Lighting Charm', # Lumos
             'Lumos Maxima', # Lumos Maxima
             'Lumos Solem Spell', # Lumos Solem
             'Muffliato Charm', # Muffliato
             'Wand-Extinguishing Charm', # Nox
             'Oculus Reparo', # Oculus Reparo
             'Peskipikski Pesternomi', # Peskipikski Pesternomi, no effect
             'Piertotum Locomotor', # Piertotum Locomotor
             'Portus', # Portus
             'Shrinking Charm', # Reducio 
             'Muggle-Repelling Charm', # Repello Muggletum
             'Revelio Charm', # Revelio
             'Tickling Charm', # Rictusempra
             'Boggart-Banishing Spell', # Riddikulus
             'Snake Summons Spell', # Serpensortia
             'Vera Verto', # Vera Verto
             'Snake-Vanishing Spell', # Vipera Evanesca
             'Levitation Charm'] # Wingardium Leviosa
C = len(dictionary)
print(C)

61


To make it smoother and more fun, we'll start by giving you the topics, $\phi$ and the proportions $\theta$ directly. I.e., you don't need to generate (yet) those from Dirichlet distribution.

Lets start by defining the 3 topics. We do this by assigning a probability for each word under of the 3 topics ($\phi$).

In [50]:
## Define word vectors (for each topic) and normalize:
phi = np.zeros( (K, C) );
# 12 spells in topic 1, 16 in topic 2, and 33 in topic 3 

phi[0] = [0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.7,
          0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
          0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01]
phi[0] *= 1/np.sum(phi[0])
phi[1] = [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.3,
          0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9,
          0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01]
phi[1] *= 1/np.sum(phi[1])
phi[2] = [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
          0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01,
          0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9]
phi[2] *= 1/np.sum(phi[2])

Notice that for topic 1, we assigned high probability to the words: Copenhagen, Madrid, Sydney, Kabul, Vienna, Brussels, Beijing, Kathmandu, Singapore, Oslo. 

Similarly, for topic 2, we assigned high probability to the words: blue, green, ...

Finaly, for topic 3, we assigned high probability to the words: model, inference, ...

Now that we have defined the probability of each word under each topic, we can follow the generative process to sample some documents. As previously mentioned, we will pre-define the distribution over topics for each document ($\theta$):

In [51]:
I = 3 ## Number of documents.

## Define topic proportion vectors (one for for each document):

# in HP Case, a 'Document' is a person, and the 'words' are the spells they use.
# We can extract the character spell proportions, and implement them here.
theta = np.zeros( (I, K) );
theta[0] = [0.6, 0.2, 0.2]
theta[1] = [1/3, 1/3, 1/3] ## This one will be an equal mix of all topics
theta[2] = [  0,   0,  1] ## This will for example only contain topic 3.

Notice that the first document has higher probability of containing words from topic 1 than topics 2 and 3. Document 2 is uniform - all topics have equal probability. Document 3 covers exclusevely topic 3.

Let us now generate the 3 documents following the generative process:

So, you assume now that we have $\boldsymbol{\theta}_i$ for all documents and $\boldsymbol{\phi}_k$ for all topics. Ancestral sampling is simply:


For each document $i$, and for each word $j=1...w_i$, do:
\begin{align}
z_{i, j} &\sim Cat(\boldsymbol{\theta}_i) \\
w_{i, j} &\sim Cat(\boldsymbol{\phi}_{z_{i,j}})
\end{align}



Can you try code it down yourself?

In [52]:
##Just to make things practical, we created this function for you
## just read it or, try it, to see what it does...
def categorical_sample(p):
    return list(np.random.multinomial(1, p)).index(1)

In [53]:
## function that receives document sizes J, theta (topic proportions) and phi (topics), 
## returns z (word-topic assignments ) and w (words)
def ancestral_sampling(J, theta, phi):
    ## Initialise
    z = np.zeros( (I, np.max(J)), dtype=int )  #NOTICE that z and w are vectors of integers!
    w = np.zeros( (I, np.max(J)), dtype=int )  
    """
    TODO: Write your code below
    """
    for i in range(I):
        for j in range(J[i]):
            z[i,j] = categorical_sample(theta[i])
            w[i,j] = categorical_sample(phi[z[i,j]])
    return z, w

Don't forget to take a look at the generated documents. Do they make sense, given the values for $\phi$ and $\theta$ given above?

In [54]:
J = [12, 9, 10] ## Vector of size I denoting how many words are in each document
z, w = ancestral_sampling(J, theta, phi)
print("w:", w)
for i in range(I):
    print("\n\nDocument: ", i)
    print("Theta: ", theta[i])
    for j in range(J[i]):
        print("Word %d: Topic assignment: %d->   %s  " % (j,  z[i, j]+1, dictionary[w[i, j]]))

w: [[23 22 21  9  3  6 10  1  3  2 37 54]
 [48  0 53  3 26 11  4  2 10  0  0  0]
 [29 47 39 59 44 47 54 32 41 29  0  0]]


Document:  0
Theta:  [0.6 0.2 0.2]
Word 0: Topic assignment: 2->   Protego totalum  
Word 1: Topic assignment: 2->   Protego Maxima  
Word 2: Topic assignment: 2->   Shield Charm  
Word 3: Topic assignment: 1->   Oppugno Jinx  
Word 4: Topic assignment: 1->   Leg-Locker Curse  
Word 5: Topic assignment: 1->   Exploding Charm  
Word 6: Topic assignment: 1->   Sectumsempra  
Word 7: Topic assignment: 1->   Blasting Curse  
Word 8: Topic assignment: 1->   Leg-Locker Curse  
Word 9: Topic assignment: 1->   Cruciatus Curse  
Word 10: Topic assignment: 3->   Engorgement Charm  
Word 11: Topic assignment: 3->   Revelio Charm  


Document:  1
Theta:  [0.33333333 0.33333333 0.33333333]
Word 0: Topic assignment: 3->   Oculus Reparo  
Word 1: Topic assignment: 1->   Killing Curse  
Word 2: Topic assignment: 3->   Muggle-Repelling Charm  
Word 3: Topic assignment: 1->   Leg-Lo

We bet you guessed what we want you to do next: having generated yourself the data, can you make an LDA model in Pyro that recovers the original parameters $\phi$ and $\theta$?

A dataset with only 3 documents is too little of course. So, let's generate 100 documents intead, by using the dirichlet distribution. In other words, we will generate:
- vectors $\theta_1 \dots \theta_I$
- a new vector $J$ that contains their size (10)


Please reuse your code to generate the dataset W. Please reuse $\phi$ as before

In [55]:
I=100
theta = np.zeros((I, K));
alpha = 0.5*np.ones(K) ## Size K dirichlet prior
J=[]

""""
TODO: Write your code here
"""
for i in range(I):
    theta[i] = np.random.dirichlet(alpha);
    J.append(10)
_, W = ancestral_sampling(J, theta, phi)

## Part 3 - Implement your own LDA model in Pyro

Let's try to implement  the model in Pyro. In order to implement the model we will need  the ```Dirichlet(alpha)``` and ```Categorical(pi)```  distributions.  We also need to marginalize the "word_topic" distribution as we did in the mixture model notebooks using ```infer={"enumerate": "parallel"}``` inside the distribution.

For a short recap, the generative process is the following:

\begin{align}
\phi_{k} & \sim Dirichlet(\alpha) \quad \textrm{Distribution of words in topic} \\
\theta_{i} & \sim Dirichlet(\beta) \quad \textrm{Distribution of topics in document} \\
& z_{i,j} \sim Categorical(\theta_{i}) \quad \textrm{A topic assignment to word $j$ in document $i$}\\
& w_{i,j}  \sim Categorical(\phi_{z_{i,j}}) \quad \textrm{A draw of word $j$ in document $i$}  \\
\end{align}

__NOTE! make sure to use the names we propose in the commented text inside the model. This is important for the inference part.__

In [56]:
# This is a fully generative model of a batch of documents.
# data is a [num_words_per_doc, num_documents] shaped array of word ids
# (specifically it is not a histogram). We assume in this simple example
# that all documents have the same number of words.

num_words = C
num_topics = K
num_docs = I
num_words_per_doc = 10

def model(data=None, batch_size=None):
    """ Make a plate of size num_topics with name "topics" and define a variable "topic_words".
          This represents the phi above. Use the equivalent of a uniform distribution for it  """
    with pyro.plate("topics", num_topics):
        topic_words = pyro.sample("topic_words", dist.Dirichlet(torch.ones(num_words) / num_words))

    """ Make two (nested) plates in here. One over documents and one over words
          Documents, called "documents":
          The plate over the documents should hold a variable "doc_topics" representing the theta above.
            Use the equivalent of a uniform distribution for it.
          
          Words, called "words":
          The plate over words, should have a topic assignment for each word (z_{i,j} above) which should be enumerated.
          The second variable "doc_words" should be the words themselves which should be drawn from the "topic_words" using the assigned z_{i,j} and the observed data.
     """
    with pyro.plate("documents", num_docs, batch_size) as ind:
        data = data[:, ind]
        doc_topics = pyro.sample("doc_topics", dist.Dirichlet(torch.ones(num_topics)/ num_topics))
        with pyro.plate("words", num_words_per_doc):
            # The word_topics variable is marginalized out during inference,
            # achieved by specifying infer={"enumerate": "parallel"} and using
            # TraceEnum_ELBO for inference. Thus we can ignore this variable in
            # the guide.
            word_topics = pyro.sample("word_topics", dist.Categorical(doc_topics), infer={"enumerate": "parallel"})
            doc_words = pyro.sample("doc_words", dist.Categorical(topic_words[word_topics]), obs=data)

    return topic_words, doc_words

In [57]:
# We can generate synthetic data directly by calling the model.
#true_topic_words, fake_data = model()
#fake_data

In [58]:
W_torch = torch.tensor(W.T).long()
W_torch.shape

torch.Size([10, 100])

## Manual+Autoguide version

In Variational Inference (VI) we define a "guide", a function similar to a model definition where we declare, for each distribution in the model, an approximating distribution "easy" to optimize. In this case, the inference can't be done using the "autoguide" as we have done in previous notebooks. The problem is that autoguide is using a multivariate normal distribution to approximate all the distributions in the model. However, some distributions (like a dirichlet) cannot be approximated using a normal. We have to use a "manual" guide in here.

Don't worry too much about this as we will learn the details on the variational inference lecture.

In [59]:
pyro.clear_param_store()

def my_local_guide(data=None, batch_size=None):
    topic_words_posterior = pyro.param(
            "topic_words_posterior",
            lambda: torch.ones(num_topics, num_words),
            constraint=constraints.positive)
    with pyro.plate("topics", num_topics):
        pyro.sample("topic_words", dist.Dirichlet(topic_words_posterior))
    
    doc_topics_posterior = pyro.param(
            "doc_topics_posterior",
            lambda: torch.ones(num_docs, num_topics),
            constraint=constraints.simplex)
    with pyro.plate("documents", num_docs, batch_size) as ind:
        pyro.sample("doc_topics", dist.Delta(doc_topics_posterior[ind], event_dim=1))
    
guide = AutoGuideList(model)
guide.add(AutoDiagonalNormal(pyro.poutine.block(model, expose=['doc_topics'])))
guide.add(my_local_guide)  # automatically wrapped in an AutoCallable

guide = my_local_guide

elbo = TraceEnum_ELBO(max_plate_nesting=3)

optim = ClippedAdam({'lr': 0.005})
svi = SVI(model, guide, optim, elbo)

# Define the number of optimization steps
n_steps = 4000

# do gradient steps
for step in range(n_steps):
    elbo = svi.step(W_torch, batch_size=32)
    if step % 100 == 0:
        #print('.', end='')
        print("[%d] ELBO: %.1f" % (step, elbo))

[0] ELBO: 4819.9
[100] ELBO: 4647.6
[200] ELBO: 4747.3
[300] ELBO: 4663.9
[400] ELBO: 4663.6
[500] ELBO: 4677.2
[600] ELBO: 4694.2
[700] ELBO: 4650.0
[800] ELBO: 4652.4
[900] ELBO: 4346.5
[1000] ELBO: 4460.6
[1100] ELBO: 4139.2
[1200] ELBO: 4152.5
[1300] ELBO: 3921.9
[1400] ELBO: 3914.5
[1500] ELBO: 3951.7
[1600] ELBO: 3955.9
[1700] ELBO: 3704.1
[1800] ELBO: 3633.5
[1900] ELBO: 3582.2
[2000] ELBO: 3669.9
[2100] ELBO: 3423.4
[2200] ELBO: 3645.1
[2300] ELBO: 3519.3
[2400] ELBO: 3645.2
[2500] ELBO: 3412.6
[2600] ELBO: 3373.9
[2700] ELBO: 3468.3
[2800] ELBO: 3262.2
[2900] ELBO: 3291.6
[3000] ELBO: 3216.9
[3100] ELBO: 3196.0
[3200] ELBO: 2894.7
[3300] ELBO: 3019.9
[3400] ELBO: 2986.1
[3500] ELBO: 2865.8
[3600] ELBO: 2838.3
[3700] ELBO: 3127.3
[3800] ELBO: 2742.8
[3900] ELBO: 2929.6


Compare estimated $\hat{\phi}$ with true values of $\phi$ used to generate the data:

In [60]:
from pyro.infer import Predictive

predictive = Predictive(model, guide=guide, num_samples=800,
                        return_sites=("topic_words",))
samples = predictive(W_torch)
samples["topic_words"].shape

torch.Size([800, 1, 3, 61])

In [61]:
# Compute the argmax of the mean of all samples of topic for each word
#   this is the most likely topic for each word in our dictionary
samples["topic_words"].mean(axis=0)[0].argmax(axis=0)

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [62]:
# Now compute the same for the real values
np.argmax(phi, axis=0)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

Notice anything interesting? Do the results match exactly? Remember that if we call "topic 1" -> "topic 2", and we call "topic 2" -> "topic 1", the resulting model is still the same! We just "renamed" the topics. This is another case of the problem of model identifiability...



Now let's compare the estimated $\hat{\theta}_{i}$ with true values of $\theta_{i}$ used to generate the data:

In [63]:
from pyro.infer import Predictive

predictive = Predictive(model, guide=guide, num_samples=800,
                        return_sites=("doc_topics",))
predictive = Predictive(model, guide=guide, num_samples=800)
samples = predictive(W_torch)

In [64]:
samples.keys()

dict_keys(['topic_words', 'doc_topics', 'word_topics', 'doc_words'])

In [65]:
doc_topics_posterior = samples["doc_topics"].mean(axis=0)
doc_topics_posterior.shape

torch.Size([1, 100, 3])

In [66]:
# Compute the argmax of the mean of all samples of topic for each document
#   this is the most likely topic for each document in our data
torch.argmax(doc_topics_posterior, axis=2)[0]

tensor([0, 1, 1, 1, 0, 2, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 2, 2, 2, 1, 0, 2,
        1, 0, 0, 1, 0, 2, 1, 2, 1, 2, 2, 2, 2, 0, 1, 0, 0, 2, 2, 2, 1, 0, 1, 0,
        2, 1, 0, 0, 2, 0, 1, 1, 2, 0, 2, 1, 0, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 2,
        2, 1, 0, 0, 0, 0, 0, 2, 1, 2, 0, 0, 0, 2, 2, 0, 2, 2, 2, 0, 1, 0, 2, 1,
        0, 0, 1, 2])

In [67]:
# Compute the same quantity for the real values
np.argmax(theta, axis=1)

array([0, 1, 0, 1, 0, 2, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 2, 2, 2, 1,
       0, 0, 1, 0, 0, 0, 0, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 0, 0, 2, 2, 0,
       1, 0, 1, 0, 2, 1, 0, 0, 2, 1, 1, 1, 2, 0, 0, 1, 1, 2, 0, 0, 1, 1,
       1, 0, 2, 1, 2, 2, 2, 1, 0, 0, 1, 1, 0, 2, 1, 2, 0, 1, 0, 2, 2, 0,
       2, 2, 2, 0, 1, 0, 2, 1, 0, 0, 1, 1])